# 01 — Ingestion & RAG Pipeline Test
Download 2 companies (AAPL, MSFT), parse, chunk, embed, store, and query.

In [ ]:
import sys
sys.path.insert(0, '..')
import config
print('Project root:', config.PROJECT_ROOT)
print('Azure available:', config.azure_available())

## 1. Download Filings (AAPL, MSFT)

In [ ]:
from src.ingestion.sec_downloader import download_all

test_tickers = {"AAPL": config.COMPANIES["AAPL"], "MSFT": config.COMPANIES["MSFT"]}
manifest = download_all(tickers=test_tickers, max_per_type=2)
print(f'\nDownloaded {len(manifest)} filings')

## 2. Parse Filings

In [ ]:
from src.ingestion.filing_parser import parse_all_filings

parsed = parse_all_filings(config.SEC_FILINGS_DIR)
for p in parsed[:2]:
    print(f"{p['ticker']} {p.get('filing_type','?')} — {len(p['sections'])} sections, {p['full_text_length']:,} chars")

## 3. Chunk

In [ ]:
from src.pipeline.chunker import chunk_all_filings, token_len

chunks = chunk_all_filings(config.SEC_FILINGS_DIR)
print(f'\nSample chunk ({chunks[0]["metadata"]["ticker"]} / {chunks[0]["metadata"]["section"]}):')
print(chunks[0]['text'][:300], '...')
print(f'Tokens: {chunks[0]["metadata"]["token_count"]}')

## 4. Embed (requires Azure OpenAI key)

In [ ]:
from src.pipeline.embedder import embed_chunks

chunks = embed_chunks(chunks)
if 'embedding' in chunks[0]:
    print(f'Embedding dim: {len(chunks[0]["embedding"])}')
else:
    print('Skipped — no Azure key. Store will work without dense vectors.')

## 5. Store in ChromaDB + BM25

In [ ]:
from src.pipeline.vector_store import HybridStore

store = HybridStore()
store.add_chunks(chunks)
print(f'Collection count: {store.collection.count()}')

## 6. Test Query

In [ ]:
query = "What are Apple's main risk factors?"

# Get query embedding if Azure is available
query_emb = None
if config.azure_available():
    from src.pipeline.embedder import embed_batch, get_client
    query_emb = embed_batch([query], get_client())[0]

results = store.search(query, query_embedding=query_emb, k=5, ticker='AAPL')

print(f'Query: {query}\n')
for i, r in enumerate(results):
    score_key = 'rerank_score' if 'rerank_score' in r else 'score'
    print(f'--- Result {i+1} (score: {r[score_key]:.4f}) ---')
    print(f'Section: {r["metadata"]["section"]} | Filing: {r["metadata"]["filing_type"]} {r["metadata"]["filing_date"]}')
    print(r['text'][:200], '...\n')